# 01 - Bernstein-Vazirani: lấy cả chuỗi bí mật trong 1 lần gọi

Bài Deutsch-Jozsa cho ta 1 bit thông tin ("hằng hay cân bằng"). Bernstein-Vazirani (BV) dùng
**đúng kỹ thuật kickback đó** nhưng lấy về **cả $n$ bit** của một chuỗi bí mật - vẫn chỉ một lần
gọi oracle. Đây là ví dụ gọn nhất cho thấy kickback không phải trò ảo thuật một lần dùng.

> **Bài toán.** Có một chuỗi bí mật $s \in \{0,1\}^n$. Oracle tính
> $$f(x) = s \cdot x \bmod 2 = (s_0x_0 \oplus s_1x_1 \oplus \dots \oplus s_{n-1}x_{n-1})$$
> Tìm $s$.

Cần trước: `00_phase_kickback_va_deutsch.ipynb`.

In [ ]:
import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit

print("qiskit:", qiskit.__version__)

sampler = SamplerV2()


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def do(qc, shots=1024, reg="c"):
    """Chạy mạch và trả về counts từ thanh ghi cổ điển tên `reg`."""
    ket_qua = sampler.run([qc], shots=shots).result()[0]
    return getattr(ket_qua.data, reg).get_counts()

## 1. Cách cổ điển: $n$ lần gọi, mỗi lần một bit

Cho $x = 100\ldots0$ thì $f(x) = s_{n-1}$. Cho $x = 010\ldots0$ thì được bit kế tiếp. Mỗi lần gọi
lấy được đúng **1 bit**, nên cần $n$ lần - và không có cách nào ít hơn, vì $n$ bit thông tin
thì phải cần $n$ lần trả lời 1 bit.

In [ ]:
def f_co_dien(s, x):
    """f(x) = s . x mod 2, với s và x là chuỗi bit."""
    return sum(int(a) * int(b) for a, b in zip(s, x)) % 2


s_that = "1011"
n = len(s_that)

tim_duoc = ""
for i in range(n):
    x = "0" * i + "1" + "0" * (n - i - 1)   # chỉ bật 1 bit
    bit = f_co_dien(s_that, x)
    tim_duoc += str(bit)
    print(f"  gọi lần {i + 1}: x={x} -> f(x)={bit}")

print(f"\ncổ điển: {n} lần gọi -> s = {tim_duoc}  (thật: {s_that})")

## 2. Oracle lượng tử cho $f(x) = s\cdot x$

Dạng bit-flip quen thuộc: $U_f\lvert x\rangle\lvert y\rangle = \lvert x\rangle\lvert y \oplus s\cdot x\rangle$.

Hiện thực cực đơn giản: với mọi vị trí $i$ mà $s_i = 1$, đặt một `cx` từ qubit $i$ vào ancilla.
Khi đó ancilla bị lật đúng $\big(\sum_{i: s_i=1} x_i\big) \bmod 2 = s\cdot x$ lần.

Chú ý quy ước little-endian: `s[-1]` (ký tự bên phải của chuỗi) ứng với **qubit 0**.

In [ ]:
def oracle_bv(s):
    """U_f cho f(x) = s.x; qubit 0..n-1 = dữ liệu, qubit n = ancilla."""
    n = len(s)
    qc = QuantumCircuit(n + 1, name="U_s")
    for i, bit in enumerate(reversed(s)):   # s[-1] <-> qubit 0
        if bit == "1":
            qc.cx(i, n)
    return qc


print("s = 1011 ->")
print(oracle_bv("1011").draw())

## 3. Mạch lượng tử: 1 lần gọi

Y hệt Deutsch-Jozsa:

1. $H$ lên tất cả qubit dữ liệu, ancilla về $\lvert -\rangle$
2. gọi oracle **một lần**
3. $H$ lên tất cả qubit dữ liệu
4. đo - ra thẳng $s$

Vì sao ra đúng $s$? Sau bước 2, kickback cho ta

$$\frac{1}{\sqrt{2^n}}\sum_x (-1)^{s\cdot x}\lvert x\rangle$$

Đây chính xác là trạng thái mà ta thu được nếu áp $H^{\otimes n}$ lên $\lvert s\rangle$.
Mà $H^{\otimes n}$ là nghịch đảo của chính nó, nên bước 3 đưa trạng thái về đúng
$\lvert s\rangle$ - đo ra là chắc chắn 100%, không phải xác suất.

In [ ]:
def bernstein_vazirani(s):
    n = len(s)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(n)                 # ancilla -> |->
    qc.h(range(n))
    qc.barrier()
    qc.compose(oracle_bv(s), inplace=True)
    qc.barrier()
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


qc = bernstein_vazirani("1011")
print(qc.draw())
print("\ncounts:", do(qc, shots=1024))

In [ ]:
# Thử nhiều chuỗi bí mật khác nhau - lần nào cũng đúng, chỉ 1 lần gọi
for s in ["1011", "0110", "1111", "0001", "10110101"]:
    c = do(bernstein_vazirani(s), shots=256)
    dung = list(c) == [s]
    print(f"  s = {s:>9s} -> đo được {list(c)[0]:>9s}  {'OK' if dung else 'SAI'}  "
          f"(cổ điển cần {len(s)} lần gọi, lượng tử 1)")

## 4. Nhìn tận trạng thái giữa chừng

Để thấy rõ "phase biến thành bit", ta chạy lại mạch nhưng **dừng trước bước đo** và in statevector.
Bỏ ancilla đi cho dễ nhìn bằng cách xét riêng $n$ qubit dữ liệu.

In [ ]:
def phan_du_lieu(sv, n):
    """Tách riêng n qubit dữ liệu khi ancilla (qubit cao nhất) đang ở |->."""
    return Statevector(sv.data[: 2 ** n] * np.sqrt(2))


s = "101"
n = len(s)

buoc = QuantumCircuit(n + 1)
buoc.x(n)
buoc.h(n)
buoc.h(range(n))
print("1) sau H: superposition đều, mỗi amplitude = 1/sqrt(8) = +0.354")
print("   ", ket(phan_du_lieu(Statevector(buoc), n)))

buoc.compose(oracle_bv(s), inplace=True)
print("\n2) sau oracle: mỗi số hạng mang dấu (-1)^(s.x) - thông tin nằm ở DẤU")
print("   ", ket(phan_du_lieu(Statevector(buoc), n)))

buoc.h(range(n))
print("\n3) sau H lần hai: interference gom hết về đúng một ket")
print("   ", ket(phan_du_lieu(Statevector(buoc), n)))
print(f"\n   -> đọc ra đúng s = {s}")
print("   (ở đây đã tách riêng phần dữ liệu; nếu in cả ancilla thì mỗi số hạng còn")
print("    bị nhân thêm dấu của |-> và amplitude giảm còn 0.250)")

## 5. Cái giá phải trả

BV thắng về **số lần gọi oracle** ($1$ thay vì $n$), nhưng oracle không miễn phí: bên trong nó
vẫn có tới $n$ cổng `cx`. Điều này rất đáng nhớ khi đọc các tuyên bố "nhanh hơn theo hàm mũ":

- Thước đo ở đây là **số truy vấn** (query complexity), không phải tổng số cổng.
- BV là bài toán được **thiết kế** để lợi thế đó lộ ra. Nó không nói gì về việc máy lượng tử
  có nhanh hơn máy thường trong các bài toán thực tế hay không.

In [ ]:
for s in ["1011", "10110101", "1" * 16]:
    qc = bernstein_vazirani(s)
    print(f"  n={len(s):>2}: số cổng = {qc.size():>3}, depth = {qc.depth():>3}, "
          f"cx = {qc.count_ops().get('cx', 0):>2}")

## Tóm tắt

| | Cổ điển | Lượng tử (BV) |
|---|---|---|
| Số lần gọi oracle | $n$ | $1$ |
| Kết quả | chắc chắn | chắc chắn (không phải xác suất) |
| Mạch | - | $H^{\otimes n}$, oracle, $H^{\otimes n}$ |

Cơ chế: oracle đập dấu $(-1)^{s\cdot x}$ lên từng số hạng (kickback), rồi $H^{\otimes n}$ -
vì là nghịch đảo của chính nó - gom tất cả về đúng ket $\lvert s\rangle$. Đây là lần đầu ta
thấy **interference** được dùng có ý: các amplitude sai triệt tiêu nhau, amplitude đúng cộng dồn lại.

### Thử tự làm

1. Chạy BV với $s = 0000$. Kết quả có hợp lý không, và oracle lúc đó là mạch gì?
2. Sửa `oracle_bv` thành dạng có thêm hằng số: $f(x) = s\cdot x \oplus 1$. Kết quả đo được
   có đổi không? Giải thích bằng global phase.
3. Đếm xem với $n = 20$ thì cổ điển cần bao nhiêu lần gọi, và thử chạy BV với $n = 20$.

Bài tiếp theo: `02_grover.ipynb`.